# MSR – RQ1: which rules is each agent PR under?

1. Load agent PRs (AIDev) and context file history (GHAgentFiles)
2. For each PR, find the context files that existed when it was opened
3. Detect which rules (from `rules.md`) those files state
4. Count PRs per rule

In [1]:
import re
from functools import cache

import pandas as pd

DATA = "../../instruction-data"
AIDEV = "hf://datasets/hao-li/AIDev@v4"

## 1. Load data

In [2]:
prs = pd.read_parquet(
    f"{AIDEV}/pull_request.parquet",
    columns=["id", "agent", "title", "state", "created_at", "merged_at", "repo_url", "html_url"],
)
prs["repo"] = prs.repo_url.str.replace("https://api.github.com/repos/", "").str.lower()
prs["created_at"] = pd.to_datetime(prs.created_at, utc=True)

changes = pd.read_csv(f"{DATA}/context_metadata.csv", low_memory=False)
changes["repo"] = changes.repository.str.lower()
changes["date"] = pd.to_datetime(changes.commit_date, unit="s", utc=True)
changes = changes[~changes.symbolic_link]  # symlinks just point to another context file

changes_by_repo = {repo: group for repo, group in changes.groupby("repo")}

print(len(prs), "PRs,", len(changes), "context file changes")

/Users/peco/School/ESS/ess-msr/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


71677 PRs, 235188 context file changes


## 2. Context files that existed when each PR was opened

In [3]:
def context_files_at(repo, when):
    """Hashes of the latest version of each context file in `repo` committed before `when`."""
    if repo not in changes_by_repo:
        return []

    history = changes_by_repo[repo]
    before = history[history.date < when]

    latest = before.sort_values("date").groupby("file_path").tail(1)
    latest = latest[latest.git_change_type != "D"]

    # a renamed file still shows its old path as alive -> drop old paths
    renamed_away = before.loc[before.git_change_type == "R", "previous_file_path"]
    latest = latest[~latest.file_path.isin(renamed_away)]

    return list(latest.file_hash.dropna())  # some renames have no stored content


prs["context_hashes"] = [context_files_at(repo, when) for repo, when in zip(prs.repo, prs.created_at)]

prs = prs[prs.context_hashes.str.len() > 0]
print(len(prs), "PRs in", prs.repo.nunique(), "repos had a context file")

23422 PRs in 1326 repos had a context file


## 3. Which rules do those files state?

Simple keyword regexes per rule (see `rules.md`). They will have false positives; we validate by hand later.

In [ ]:
MANAGER = r"`?(pnpm|yarn|bun|uv|poetry)`?(?:\s*\([^)]*\))?"  # optional "(v9+)" after the name
PACKAGE_MANAGER_PHRASES = [
    rf"\b(?:use|using|prefer)\s+{MANAGER}\s+(?:as|for)\s+(?:the\s+|our\s+|your\s+)?(?:package|dependenc|js\b|javascript|node|python)",  # use pnpm as the package manager
    rf"\b(?:use|using)\s+{MANAGER}\s+(?:only|exclusively)\b",  # use yarn exclusively
    rf"\bonly\s+use\s+{MANAGER}",  # only use bun
    rf"\b(?:use|run)\s+`?(pnpm|yarn|bun|uv|poetry)\s+(?:install|add|sync)\b",  # use `pnpm install`
    rf"package\s+manager\b[\s*:`_-]*(?:is\s+|use\s+)?{MANAGER}",  # **Package manager**: uv
    rf"package\s+manager,\s+use\s+{MANAGER}",  # ...need a package manager, use pnpm
    rf"{MANAGER}[\s*:_-]+package\s+manager",  # **PNPM**: Package manager
    rf"{MANAGER}\s+is\s+(?:the|our)\b[^.\n]{{0,20}}package\s+manager",  # pnpm is the project's package manager
]

# `^(?!.*X)` means "the line does not contain X"
RULE_PATTERNS = {
    "commit_format": r"^(?!.*\b(do not|don't|never|consider|encouraged|optional|when possible|if possible|when reasonable)\b).*conventional commits?",
    "required_tests": (
        r"\b(must|should|always|required?)\b[^.\n]*\b(add|write|include|accompanied)\b[^.\n]{0,30}\btests?\b"
        r"|\b(add|write|include)\b[^.\n]{0,30}\btests?\b[^.\n]*\b(whenever|for (any|every|all|new))\b"
    ),
    "package_manager": "|".join(PACKAGE_MANAGER_PHRASES),
    "changelog": r"^(?!.*\brelease).*\b(update|add|edit)\b[^.\n]{0,40}\bchangelog\b",
}


@cache
def read_file(file_hash):
    with open(f"{DATA}/context_content/{file_hash}", encoding="utf-8", errors="replace") as f:
        return f.read()


def context_text(hashes):
    return "\n".join(read_file(h) for h in hashes)


def stated_rules(hashes):
    text = context_text(hashes)
    return [
        rule for rule, pattern in RULE_PATTERNS.items()
        if re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
    ]


prs["rules"] = prs.context_hashes.apply(stated_rules)

## 4. How many PRs fall under each rule?

In [5]:
pr_rule = prs.explode("rules", ignore_index=True).dropna(subset=["rules"])

pr_rule.groupby("rules").agg(prs=("id", "count"), repos=("repo", "nunique"))

,prs,repos
rules,,
changelog,1334,54
commit_format,5228,150
package_manager,2733,194
required_tests,7436,476


In [6]:
pd.crosstab(pr_rule.rules, pr_rule.agent)

agent,Claude_Code,Copilot,Cursor,Devin,Google_Jules,OpenAI_Codex
rules,,,,,,
changelog,53,426,263,53,15,524
commit_format,210,797,277,184,0,3760
package_manager,152,758,780,437,0,606
required_tests,435,4532,1102,214,18,1135


## 5. Sanity check: do the regexes catch real rules?

Print random lines that matched each rule. If many are not real rules, tighten the pattern in step 3 and rerun.

In [ ]:
def matching_lines(rule, n=20):
    hashes = pr_rule.loc[pr_rule.rules == rule, "context_hashes"].explode().unique()

    lines = []
    for file_hash in hashes:
        for line in read_file(file_hash).splitlines():
            if re.search(RULE_PATTERNS[rule], line, re.IGNORECASE):
                lines.append(line.strip())

    lines = pd.Series(lines).drop_duplicates()
    return lines.sample(min(n, len(lines)), random_state=0)


for rule in RULE_PATTERNS:
    print(f"\n===== {rule} =====")
    for line in matching_lines(rule):
        print("-", line[:200])

## 6. Which files did each PR change?

`pr_commit_details` has one row per (commit, changed file). We keep the file names and which ones were newly added. `pr_commits` gives the commit messages.

In [ ]:
files = pd.read_parquet(f"{AIDEV}/pr_commit_details.parquet", columns=["pr_id", "filename", "status"])
files = files[files.pr_id.isin(prs.id)].dropna()

changed_files = files.groupby("pr_id").filename.apply(set)
prs["changed_files"] = prs.id.map(changed_files)

added_files = files[files.status == "added"].groupby("pr_id").filename.apply(set)
prs["added_files"] = [added_files.get(pr_id, set()) for pr_id in prs.id]

commits = pd.read_parquet(f"{AIDEV}/pr_commits.parquet", columns=["pr_id", "message"])
commits = commits[commits.pr_id.isin(prs.id)].dropna()
commit_messages = commits.groupby("pr_id").message.apply(lambda msgs: [m.split("\n")[0] for m in msgs])
prs["commit_messages"] = [commit_messages.get(pr_id, []) for pr_id in prs.id]

print(prs.changed_files.isna().sum(), "of", len(prs), "PRs have no file info")
prs[["title", "changed_files"]].head()

## 7. Checkers

Each checker gets one (PR, rule) row and returns `not_applicable`, `complied` or `violated`.

### Commit / PR title format

Complied if the **PR title or any commit message** follows Conventional Commits.
A leading `[tag]` is ignored (some repos ask for e.g. `[alpha_factory] feat: ...`).
This is lenient on purpose: we would rather miss a violation than invent one.

In [ ]:
# "type(scope)!: description" - the spec allows any type word (repos add their own, e.g. "k8s:")
CONVENTIONAL_TITLE = r"^[\w,-]+(\(.+\))?!?: .+"


def follows_format(text):
    text = re.sub(r"^\[[^\]]*\]\s*", "", text.strip())
    return re.match(CONVENTIONAL_TITLE, text, re.IGNORECASE) is not None


def check_commit_format(pr):
    if follows_format(pr.title) or any(follows_format(m) for m in pr.commit_messages):
        return "complied"
    return "violated"


(
    check_commit_format(pd.Series({"title": "[alpha_factory] add x", "commit_messages": ["feat: add x"]})),  # complied
    check_commit_format(pd.Series({"title": "Add sidebar", "commit_messages": ["Initial plan"]})),  # violated
)

### Package manager

1. Read which manager(s) the context file asks for (e.g. `pnpm`).
2. The rule applies only if the PR changed a lockfile of the same ecosystem (JS or Python).
3. Violated if the PR **added a new** lockfile of a different manager (e.g. a new `package-lock.json` when pnpm is required).
   Editing a wrong lockfile that already existed is not counted: some sub-packages legitimately use another manager.
4. Lockfiles inside `test/`, `fixtures/`, `examples/` folders are ignored (sample projects).

In [ ]:
LOCKFILE_MANAGER = {
    "package-lock.json": "npm",
    "pnpm-lock.yaml": "pnpm",
    "yarn.lock": "yarn",
    "bun.lock": "bun",
    "bun.lockb": "bun",
    "uv.lock": "uv",
    "poetry.lock": "poetry",
}
ECOSYSTEM = {"npm": "js", "pnpm": "js", "yarn": "js", "bun": "js", "uv": "python", "poetry": "python"}


IGNORED_FOLDERS = {"test", "tests", "fixtures", "__fixtures__", "example", "examples"}


def required_managers(text):
    found = set()
    for phrase in PACKAGE_MANAGER_PHRASES:
        found |= {name.lower() for name in re.findall(phrase, text, re.IGNORECASE)}
    return found


def in_ignored_folder(path):
    folders = path.lower().split("/")[:-1]
    return any(folder in IGNORED_FOLDERS for folder in folders)


def lockfile_managers(file_paths, ecosystems):
    managers = {LOCKFILE_MANAGER.get(path.split("/")[-1]) for path in file_paths if not in_ignored_folder(path)}
    return {m for m in managers if m and ECOSYSTEM[m] in ecosystems}


def check_package_manager(pr):
    required = required_managers(context_text(pr.context_hashes))
    ecosystems = {ECOSYSTEM[m] for m in required}

    changed = lockfile_managers(pr.changed_files, ecosystems)
    added = lockfile_managers(pr.added_files, ecosystems)

    if added - required:
        return "violated"
    if changed and changed <= required:
        return "complied"
    return "not_applicable"


(
    required_managers("Use pnpm as the package manager. Run `uv sync` first."),  # -> {'pnpm', 'uv'}
    required_managers("Use `yarn validate` before pushing"),  # -> set(): just a command
)

### Required tests

1. Applies if the PR changed at least one **source file** (code that is not a test).
2. Complied if it also changed at least one **test file**, otherwise violated.

In [ ]:
SOURCE_EXTENSIONS = {
    ".py", ".js", ".jsx", ".ts", ".tsx", ".go", ".rs", ".java", ".kt", ".rb", ".php",
    ".cs", ".c", ".cpp", ".h", ".swift", ".scala", ".dart", ".vue", ".svelte",
}

TEST_PATH = (
    r"(^|/)(tests?|__tests__|specs?)/"  # inside a test folder:   tests/foo.py, src/test/Foo.java
    r"|(^|/)test_[^/]*$"  # test_ prefix:            test_foo.py
    r"|[._-](test|spec)s?\.[a-z]+$"  # test/spec suffix:        foo_test.go, foo.test.ts, foo.spec.js
    r"|[a-z0-9]Tests?\.[a-z]+$"  # CamelCase suffix:        FooTest.java, FooTests.cs
)


def is_test_file(path):
    return re.search(TEST_PATH, path) is not None


def is_source_file(path):
    extension = "." + path.split(".")[-1].lower() if "." in path else ""
    return extension in SOURCE_EXTENSIONS and not is_test_file(path)


def check_required_tests(pr):
    if not any(is_source_file(f) for f in pr.changed_files):
        return "not_applicable"
    if any(is_test_file(f) for f in pr.changed_files):
        return "complied"
    return "violated"


[(f, is_test_file(f), is_source_file(f)) for f in ["src/app.py", "tests/test_app.py", "pkg/x_test.go", "src/FooTest.java", "README.md"]]

### Changelog

1. Applies if the PR changed at least one source file.
2. Complied if it changed a changelog file (`CHANGELOG.md`, `docs/changelog/...`) or a changeset (`.changeset/*.md`), otherwise violated.

In [ ]:
CHANGELOG_PATH = r"changelog|(^|/)\.changeset/|(^|/)changes\.md$"


def check_changelog(pr):
    if not any(is_source_file(f) for f in pr.changed_files):
        return "not_applicable"
    if any(re.search(CHANGELOG_PATH, f, re.IGNORECASE) for f in pr.changed_files):
        return "complied"
    return "violated"

### Label every (PR, rule) pair

In [ ]:
CHECKERS = {
    "commit_format": check_commit_format,
    "package_manager": check_package_manager,
    "required_tests": check_required_tests,
    "changelog": check_changelog,
}

pr_rule = prs.explode("rules", ignore_index=True).dropna(subset=["rules"])
pr_rule = pr_rule[pr_rule.rules.isin(CHECKERS) & pr_rule.changed_files.notna()]

pr_rule["label"] = [CHECKERS[row.rules](row) for row in pr_rule.itertuples()]
pd.crosstab(pr_rule.rules, pr_rule.label)

## 8. RQ1: violation rate

Violation rate = violated / (complied + violated). `not_applicable` rows are left out.

`repo_averaged_rate` = mean of each repo's own rate, so a repo with 3,000 PRs counts the same as one with 10.

In [ ]:
checked = pr_rule[pr_rule.label != "not_applicable"].copy()
checked["violated"] = checked.label == "violated"

summary = checked.groupby("rules").agg(
    prs=("id", "count"),
    repos=("repo", "nunique"),
    violation_rate=("violated", "mean"),
)

per_repo_rate = checked.groupby(["rules", "repo"]).violated.mean()
summary["repo_averaged_rate"] = per_repo_rate.groupby("rules").mean()
summary.round(2)

In [ ]:
per_agent_repo = checked.groupby(["rules", "agent", "repo"]).violated.mean()
by_agent = pd.DataFrame({
    "repo_averaged_rate": per_agent_repo.groupby(["rules", "agent"]).mean(),
    "prs": checked.groupby(["rules", "agent"]).size(),
    "repos": per_agent_repo.groupby(["rules", "agent"]).size(),
})
by_agent.round(2)

## 9. Hand check: package_manager violations

Open each PR link, look at the added lockfile, and fill the `correct` column in the CSV with `yes` / `no`.

In [ ]:
def rule_line(hashes):
    for line in context_text(hashes).splitlines():
        if re.search(RULE_PATTERNS["package_manager"], line, re.IGNORECASE):
            return line.strip()


violations = pr_rule[(pr_rule.rules == "package_manager") & (pr_rule.label == "violated")]

to_check = pd.DataFrame({
    "pr_url": violations.html_url,
    "agent": violations.agent,
    "rule_line": violations.context_hashes.apply(rule_line),
    "required": violations.context_hashes.apply(lambda h: sorted(required_managers(context_text(h)))),
    "added_lockfiles": violations.added_files.apply(lambda files: [f for f in files if f.split("/")[-1] in LOCKFILE_MANAGER]),
    "correct": "",
})
to_check.to_csv("../../package_manager_violations.csv", index=False)

pd.set_option("display.max_colwidth", 120)
to_check